# Readable queries with CTEs

Practice notebook for the datafromzero SQL lesson **Readable queries with CTEs**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Write the first step on its own

In [ ]:
con.sql("""
SELECT CAST(ordered_at AS DATE) AS order_day, store_id, COUNT(*) AS orders
FROM orders
GROUP BY CAST(ordered_at AS DATE), store_id
ORDER BY order_day, store_id
LIMIT 5
""").show()

### Step 2. Name it

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS order_day, store_id, COUNT(*) AS orders
  FROM orders
  GROUP BY CAST(ordered_at AS DATE), store_id
)
SELECT * FROM daily
ORDER BY order_day, store_id
LIMIT 5
""").show()

### Step 3. Check the grain and the row count

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS order_day, store_id, COUNT(*) AS orders
  FROM orders
  GROUP BY CAST(ordered_at AS DATE), store_id
)
SELECT COUNT(*) AS store_days, SUM(orders) AS orders
FROM daily
""").show()

### Step 4. Add the second step

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS order_day, store_id, COUNT(*) AS orders
  FROM orders
  GROUP BY CAST(ordered_at AS DATE), store_id
),
average_day AS (
  SELECT store_id, COUNT(*) AS days, ROUND(AVG(orders), 2) AS avg_orders
  FROM daily
  GROUP BY store_id
)
SELECT * FROM average_day
ORDER BY store_id
""").show()

### Step 5. Debug by looking at one step

In [ ]:
con.sql("""
WITH daily AS (
  SELECT CAST(ordered_at AS DATE) AS order_day, store_id, COUNT(*) AS orders
  FROM orders
  GROUP BY CAST(ordered_at AS DATE), store_id
)
SELECT MIN(order_day) AS first_day, COUNT(*) AS days
FROM daily
WHERE store_id = 3
""").show()

### Step 6. Verify

In [ ]:
con.sql("""
SELECT store_id, ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT CAST(ordered_at AS DATE)), 2) AS avg_orders
FROM orders
GROUP BY store_id
ORDER BY store_id
""").show()

## Exercise 1. Busy days of each store

A busy day is a store-day with 6 or more orders. The `daily` step is given, and so is the start of `busy_days`. Replace the comment with the lines that keep only the busy store-days and make one group per store. The cell gives an error until you do.

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    WITH daily AS (
      SELECT CAST(ordered_at AS DATE) AS order_day, store_id, COUNT(*) AS orders
      FROM orders
      GROUP BY CAST(ordered_at AS DATE), store_id
    ),
    busy_days AS (
      SELECT store_id, COUNT(*) AS busy_days
      FROM daily
      -- add the missing lines here
    )
    SELECT * FROM busy_days
    ORDER BY store_id
    """).show()
except Exception as e:
    print(e)


## Exercise 2. The average day for each payment method

Completed orders only. Before you run anything, write one sentence in the first cell: the grain of each step. Then build two CTEs: the first counts completed orders for each payment method on each day, the second averages them for each payment method.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

A colleague writes the final step first and the daily step last. It gives an error on purpose: read the message, then write a query that works.

In [ ]:
# This cell can show an error (on purpose, or until you finish it); the notebook prints it and keeps going.
try:
    con.sql("""
    WITH average_day AS (
      SELECT store_id, ROUND(AVG(orders), 2) AS avg_orders
      FROM daily
      GROUP BY store_id
    ),
    daily AS (
      SELECT CAST(ordered_at AS DATE) AS order_day, store_id, COUNT(*) AS orders
      FROM orders
      GROUP BY CAST(ordered_at AS DATE), store_id
    )
    SELECT * FROM average_day
    """).show()
except Exception as e:
    print(e)


In [ ]:
# Your corrected query: